In [ ]:
# --- UPGRADE: TRAINING ON THE TWO-FILE PUBLIC DATABASE ---
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

print("📥 Loading the Train and Test databases...")

try:
    # 1. Load both files separately
    train_df = pd.read_csv("financial_transaction_train1.csv")
    test_df = pd.read_csv("financial_transaction_test1.csv")

    # --- IMPORTANT: Change these if your CSV columns have different names! ---
    text_column = 'Transaction_Text'       # The column with the bank narration
    category_column = 'Label'  # The column with the category (Food, Travel, etc.)
    # -------------------------------------------------------------------------

    # 2. Set up the Study Material (Train)
    X_train = train_df[text_column].astype(str)
    y_train = train_df[category_column].astype(str)

    # 3. Set up the Final Exam (Test)
    X_test = test_df[text_column].astype(str)
    y_test = test_df[category_column].astype(str)

    # 4. Build the NLP + Machine Learning Pipeline
    ml_pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer(stop_words='english', max_features=2000)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # 5. Train the Model on the Training Data
    print(f"🧠 Training the Random Forest on {len(X_train)} public records...")
    ml_pipeline.fit(X_train, y_train)

    # 6. Evaluate the Model on the Testing Data
    print("🔍 Grading the model on the hidden test data...")
    y_pred = ml_pipeline.predict(X_test)

    # 7. Print the Professional Results
    print("-" * 50)
    print(f"✅ PUBLIC MODEL ACCURACY: {accuracy_score(y_test, y_pred) * 100:.2f}%")
    print("-" * 50)
    print("\n📊 DETAILED CLASSIFICATION REPORT:\n")
    print(classification_report(y_test, y_pred, zero_division=0))

except FileNotFoundError as e:
    print(f"❌ ERROR: Could not find the file. {e}")
    print("Please make sure BOTH files are uploaded to the Colab folder!")
except KeyError as e:
    print(f"❌ ERROR: Column name mismatch. {e}")
    print(f"Open your CSV and check the exact spelling of the text and label columns.")

📥 Loading the Train and Test databases...
🧠 Training the Random Forest on 12038 public records...
🔍 Grading the model on the hidden test data...
--------------------------------------------------
✅ PUBLIC MODEL ACCURACY: 97.61%
--------------------------------------------------

📊 DETAILED CLASSIFICATION REPORT:

                   precision    recall  f1-score   support

       Automobile       0.72      0.53      0.61        34
Bills & Utilities       1.00      0.99      1.00       108
  Cash Withdrawal       1.00      1.00      1.00        50
Credit Card Bills       1.00      1.00      1.00         6
              EMI       1.00      1.00      1.00       171
        Education       1.00      1.00      1.00        50
    Entertainment       0.96      0.90      0.93        83
             Food       0.96      0.98      0.97       243
        Groceries       0.97      0.99      0.98       199
           Income       1.00      1.00      1.00        50
       Investment       1.00      1

In [ ]:
# --- UPGRADED CELL 2: DYNAMIC SELF-TRANSFER & RENT OVERRIDE ---
import pandas as pd

# 1. Load your personal bank statement
MY_BANK_STATEMENT = "acc_statement.csv"
my_df = pd.read_csv(MY_BANK_STATEMENT)

print(f"📄 Reading your personal bank statement: '{MY_BANK_STATEMENT}'...")

# ==========================================
# 👤 DYNAMIC INPUT: Entity Resolution
# ==========================================
print("\n" + "="*50)
current_user_name = input("👉 Enter the Account Holder's Name (for Self Transfers): ").strip().upper()
landlord_name = input("👉 Enter the Landlord/Owner's Name (for Rent): ").strip().upper()
print("="*50 + "\n")

# 2. Let the AI predict the categories first
print("🤖 AI is analyzing and categorizing all transactions...")
my_df['ML_Predicted_Category'] = ml_pipeline.predict(my_df['Narration'].astype(str))

# ==========================================
# 🛑 THE OVERRIDE LOGIC (Self Transfer & Rent)
# ==========================================
# Override 1: Self Transfers
if current_user_name:
    is_self = my_df['Narration'].str.upper().str.contains(current_user_name, na=False)
    my_df.loc[is_self, 'ML_Predicted_Category'] = 'Self Transfer'
    print(f"🔄 Intercepted {is_self.sum()} transactions for '{current_user_name}' -> Marked as 'Self Transfer'.")

# Override 2: Rent
if landlord_name:
    is_rent = my_df['Narration'].str.upper().str.contains(landlord_name, na=False)
    my_df.loc[is_rent, 'ML_Predicted_Category'] = 'Rent'
    print(f"🏠 Intercepted {is_rent.sum()} transactions for '{landlord_name}' -> Marked as 'Rent'.")
# ==========================================

# 3. Clean up the formatting (Debit/Credit logic)
if 'Withdrawal Amt.' in my_df.columns and 'Deposit Amt.' in my_df.columns:
    my_df['Withdrawal Amt.'] = my_df['Withdrawal Amt.'].fillna(0)
    my_df['Deposit Amt.'] = my_df['Deposit Amt.'].fillna(0)

    def standardize_amount(row):
        return (row['Deposit Amt.'], 'Credit') if row['Deposit Amt.'] > 0 else (row['Withdrawal Amt.'], 'Debit')

    my_df[['Amount', 'Type']] = my_df.apply(standardize_amount, axis=1, result_type='expand')

    # Filter to only show your expenses (Debits)
    final_expenses = my_df[my_df['Type'] == 'Debit'].copy()
else:
    final_expenses = my_df.copy()

# 4. Export and Display
output_filename = "ML_Categorized_Statement.csv"
final_expenses.to_csv(output_filename, index=False)

print(f"\n🎉 SUCCESS! Your ML-categorized data is saved as '{output_filename}'.")
print("👇 Preview of your AI's Predictions:")

# Display the most relevant columns
columns_to_show = ['Date', 'Narration', 'Amount', 'ML_Predicted_Category'] if 'Amount' in final_expenses.columns else ['Narration', 'ML_Predicted_Category']
display(final_expenses[columns_to_show].head(20))

📄 Reading your personal bank statement: 'acc_statement.csv'...

👉 Enter the Account Holder's Name (for Self Transfers): Vishal sharma
👉 Enter the Landlord/Owner's Name (for Rent): bharathy mohan

🤖 AI is analyzing and categorizing all transactions...
🔄 Intercepted 50 transactions for 'VISHAL SHARMA' -> Marked as 'Self Transfer'.
🏠 Intercepted 5 transactions for 'BHARATHY MOHAN' -> Marked as 'Rent'.

🎉 SUCCESS! Your ML-categorized data is saved as 'ML_Categorized_Statement.csv'.
👇 Preview of your AI's Predictions:


,Date,Narration,Amount,ML_Predicted_Category
0,01/01/25,UPI-VIJETHA SUPERMARKETS-VIJETHASUPERMARKETSP....,229.00,Groceries
1,02/01/25,UPI-MR ASHOK KUMAR PER-Q819149057@YBL-YESB0Y...,33.00,Local Vendor UPI
2,02/01/25,UPI-GUNDA REVATHI-PAYTMQR617Q2C@PTYS-YESB0PTMU...,40.00,Local Vendor UPI
3,02/01/25,UPI-KATTAMURI CHAKRADHAR-9177282316-2@YBL-UBIN...,47.00,Local Vendor UPI
4,02/01/25,UPI-M S SAI SWAPNA STATI-EAZYPAY.NTB1100072090...,80.00,Food
6,02/01/25,UPI-VISHAL SHARMA-8134857123@AXL-ICIC0000411-8...,3000.00,Self Transfer
7,03/01/25,UPI-VISHAL SHARMA-8134857123@AXL-ICIC0000411-4...,6500.00,Self Transfer
8,04/01/25,UPI-DANDUGULA SAMPATH-9381079836-3@IBL-SBIN00...,65.00,Local Vendor UPI
9,04/01/25,UPI-JOGU KRISHNA-PAYTMQR5Z0NDK@PTYS-YESB0PTMUP...,65.00,Local Vendor UPI
10,06/01/25,UPI-MR ASHOK KUMAR PER-Q160583297@YBL-YESB0Y...,33.00,Local Vendor UPI


In [ ]:
# --- THE ULTIMATE MULTI-YEAR DASHBOARD (TABLE + BAR + CLEAN PIE) ---
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as path_effects

print("⏳ Loading Multi-Year Financial Engine...")

try:
    # 1. Load the data
    df = pd.read_csv("ML_Categorized_Statement.csv")

    if 'Amount' not in df.columns or 'Date' not in df.columns:
        raise ValueError("Missing 'Amount' or 'Date' column in your CSV.")

    # 🛑 EXCLUDE SELF TRANSFERS
    if 'ML_Predicted_Category' in df.columns:
        df = df[df['ML_Predicted_Category'] != 'Self Transfer']

    # 2. Fix Dates to handle Multiple Years (e.g., Jan 2025, Jan 2026)
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)

    # Create a "Month-Year" column and sort properly
    df['Month_Year'] = df['Date'].dt.strftime('%b %Y') # e.g., 'Jan 2025'
    df = df.sort_values('Date')

    # Get unique Month-Years in chronological order
    unique_periods = df['Month_Year'].dropna().unique().tolist()

    # 3. Create the Output area
    plot_output = widgets.Output()

    # 4. The Visual & Tabular Engine
    def draw_dashboard(Selected_Periods):
        with plot_output:
            clear_output(wait=True)

            temp_df = df.copy()

            # Filter based on selection
            if 'All Periods' not in Selected_Periods and len(unique_periods) > 0:
                temp_df = temp_df[temp_df['Month_Year'].isin(Selected_Periods)]

            if temp_df.empty:
                print("⚠️ No transactions found for the selected period(s)!")
                return

            # --- PART 1: THE DATA TABLE ---
            print("\n" + "="*50)
            print(" 📋 CATEGORY-WISE SPEND TABLE")
            print("="*50)

            # Create a pivot table: Categories as Rows, Month-Year as Columns
            pivot_table = pd.pivot_table(
                temp_df,
                values='Amount',
                index='ML_Predicted_Category',
                columns='Month_Year',
                aggfunc='sum',
                fill_value=0
            )

            # Add a Grand Total column
            pivot_table['Total Spent (₹)'] = pivot_table.sum(axis=1)
            pivot_table = pivot_table.sort_values('Total Spent (₹)', ascending=False)

            # Format numbers to look like currency
            formatted_table = pivot_table.map(lambda x: f"₹ {x:,.2f}")
            display(formatted_table)

            grand_total = pivot_table['Total Spent (₹)'].sum()
            print(f"\n💰 OVERALL TOTAL SPENT: ₹ {grand_total:,.2f} | 📄 Total Transactions: {len(temp_df)}")
            print("="*50 + "\n")

            # --- PART 2: THE VISUALIZATIONS ---
            # Group by category for the charts
            category_totals = temp_df.groupby('ML_Predicted_Category')['Amount'].sum().reset_index()
            category_totals = category_totals.sort_values(by='Amount', ascending=False)

            # Use a slightly wider figure to accommodate the Side Legend beautifully
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

            periods_text = ", ".join(Selected_Periods) if 'All Periods' not in Selected_Periods else "All Periods"
            fig.suptitle(f'Financial Overview: {periods_text}', fontsize=24, fontweight='bold', y=1.05)

            # --------------------------------
            # CHART 1: BAR CHART (Left Side)
            # --------------------------------
            sns.barplot(x='Amount', y='ML_Predicted_Category', data=category_totals, ax=ax1, palette='viridis')
            ax1.set_title('Total Spent per Category (₹)', fontsize=18, fontweight='bold', pad=15)
            ax1.set_xlabel('Amount in INR (₹)', fontsize=14)
            ax1.set_ylabel('')
            ax1.grid(axis='x', linestyle='--', alpha=0.7)

            # Add exact ₹ numbers to the bars
            for p in ax1.patches:
                ax1.annotate(f'₹{p.get_width():,.0f}',
                             (p.get_width(), p.get_y() + p.get_height() / 2.),
                             ha='left', va='center', xytext=(5, 0), textcoords='offset points', fontsize=12)

            # --------------------------------
            # CHART 2: CLEAN PIE CHART (Right Side)
            # --------------------------------
            colors = sns.color_palette("husl", len(category_totals))

            # Draw Pie Chart hiding percentages less than 2.5% to avoid clutter
            wedges, texts, autotexts = ax2.pie(
                category_totals['Amount'],
                autopct=lambda pct: f"{pct:.1f}%" if pct > 2.5 else "",
                startangle=140,
                colors=colors,
                wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2),
                textprops=dict(fontweight='bold', color='white', fontsize=12)
            )

            # Make the percentage text look sharp with a shadow
            for autotext in autotexts:
                autotext.set_path_effects([path_effects.withStroke(linewidth=2, foreground='black')])

            # Build the Side Legend
            legend_labels = []
            for _, row in category_totals.iterrows():
                pct = (row['Amount'] / grand_total) * 100
                legend_labels.append(f"{row['ML_Predicted_Category']} ({pct:.1f}%) : ₹ {row['Amount']:,.2f}")

            # Place legend outside the chart
            ax2.legend(
                wedges, legend_labels, title="Category Breakdown", title_fontsize=14,
                loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=12, frameon=True, shadow=True, edgecolor='black'
            )

            ax2.set_title('Expense Distribution (%)', fontsize=18, fontweight='bold', pad=15)
            ax2.text(0, 0, f"Total Spent\n₹ {grand_total:,.0f}", ha='center', va='center', fontsize=16, fontweight='bold')

            # plt.tight_layout() ensures the side legend doesn't get cut off
            plt.tight_layout(rect=[0, 0, 0.85, 1])
            plt.show()

    # 5. Interactive UI (Now showing Month-Years)
    period_widget = widgets.SelectMultiple(
        options=['All Periods'] + unique_periods,
        value=['All Periods'],
        description='Periods:',
        rows=min(6, len(unique_periods) + 1)
    )

    print("🎛️ DASHBOARD READY! Select a time period below to view the Data Table and Charts.")
    widgets.interactive(draw_dashboard, Selected_Periods=period_widget)
    display(period_widget)
    display(plot_output)

except Exception as e:
    print(f"❌ ERROR: {e}")

⏳ Loading Multi-Year Financial Engine...
🎛️ DASHBOARD READY! Select a time period below to view the Data Table and Charts.


SelectMultiple(description='Periods:', index=(0,), options=('All Periods', 'Jan 2025', 'Feb 2025', 'Mar 2025',…

Output()

In [ ]:
# --- PHASE 6: PRESCRIPTIVE GOAL-SEEKING OPTIMIZER (AVERAGE & DIVERSIFIED) ---
import pandas as pd
import pulp
from IPython.display import display

print("🎯 Initializing the Upgraded AI Budget Optimizer...")

try:
    df = pd.read_csv("ML_Categorized_Statement.csv")
    df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

    ignore_list = ['Income', 'Investment', 'Cash Withdrawal', 'Self Transfer']
    opt_df = df[~df['ML_Predicted_Category'].isin(ignore_list)].copy()

    # 🛠️ THE FIX: CALCULATE TRUE MONTHLY AVERAGE 🛠️
    # 1. Make sure we have a Month_Year column
    if 'Month_Year' not in opt_df.columns:
        opt_df['Date'] = pd.to_datetime(opt_df['Date'], format='mixed', dayfirst=True)
        opt_df['Month_Year'] = opt_df['Date'].dt.strftime('%b %Y')

    # 2. Count how many unique months are in the dataset
    num_months = opt_df['Month_Year'].nunique()
    if num_months == 0: num_months = 1

    # 3. Divide the Total Sum by the Number of Months
    monthly_avg = opt_df.groupby('ML_Predicted_Category')['Amount'].sum() / num_months
    categories = monthly_avg.index.tolist()

    print("\n" + "="*50)
    target_input = input("💰 Enter your monthly savings target (₹): ").strip()
    savings_target = float(target_input) if target_input.replace('.', '', 1).isdigit() else 5000
    print("="*50 + "\n")

    prob = pulp.LpProblem("Budget_Optimization", pulp.LpMinimize)
    cut_vars = pulp.LpVariable.dicts("Cut", categories, lowBound=0, cat='Continuous')

    pain_weights = {c: 1 if c in ['Food', 'Shopping', 'Entertainment', 'Travel']
                    else (10 if c in ['Groceries', 'Medical', 'Bills & Utilities'] else 5) for c in categories}

    prob += pulp.lpSum([pain_weights[cat] * cut_vars[cat] for cat in categories])
    prob += pulp.lpSum([cut_vars[cat] for cat in categories]) == savings_target

    max_savings = 0
    for cat in categories:
        val = monthly_avg[cat]

        if cat in ['Rent', 'EMI', 'Credit Card Bills', 'Education']:
            category_max_cut = 0
        elif cat in ['Groceries', 'Medical', 'Bills & Utilities', 'Automobile']:
            category_max_cut = val * 0.15
        else:
            category_max_cut = val * 0.40

        max_savings += category_max_cut

        # Diversification limit (Max 60% of target from one category)
        diversified_limit = savings_target * 0.70
        actual_limit = min(category_max_cut, diversified_limit)

        prob += cut_vars[cat] <= actual_limit

    if savings_target > max_savings:
        print(f"⚠️ WARNING: Saving ₹ {savings_target:,.2f} is mathematically impossible without breaking essential constraints.")
        print(f"The absolute maximum you can safely save right now is ₹ {max_savings:,.2f}.")
    else:
        prob.solve()
        if pulp.LpStatus[prob.status] == 'Optimal':
            print(f"✅ SUCCESS! AI found a DIVERSIFIED path to save ₹ {savings_target:,.2f} per month:\n")

            results = []
            for cat in categories:
                cut_amt = cut_vars[cat].varValue
                if cut_amt > 0:
                    results.append({
                        "Category": cat,
                        "Average Monthly Spend": f"₹ {monthly_avg[cat]:,.2f}",
                        "Suggested Monthly Cut": f"₹ {cut_amt:,.2f}",
                        "New Target Budget": f"₹ {monthly_avg[cat] - cut_amt:,.2f}"
                    })
            display(pd.DataFrame(results))
        else:
            print("❌ Optimization failed. The constraints might be too tight for this specific target.")

except Exception as e:
    print(f"❌ Error: {e}")

🎯 Initializing the Upgraded AI Budget Optimizer...

💰 Enter your monthly savings target (₹): 5000

✅ SUCCESS! AI found a DIVERSIFIED path to save ₹ 5,000.00 per month:



,Category,Average Monthly Spend,Suggested Monthly Cut,New Target Budget
0,Automobile,₹ 122.22,₹ 18.33,₹ 103.89
1,Food,"₹ 3,847.28","₹ 1,538.91","₹ 2,308.37"
2,Local Vendor UPI,"₹ 15,778.81","₹ 2,643.11","₹ 13,135.70"
3,Shopping,₹ 888.10,₹ 355.24,₹ 532.86
4,Travel,"₹ 1,111.00",₹ 444.40,₹ 666.60
